# ⚡ Notebook 2: Optimización de Modelos Lineales (Perceptrón)

Una vez generado y validado nuestro conjunto de datos **MyDoodles**, entramos en la fase de modelado. En este entorno de experimentación, implementaremos el algoritmo **Perceptrón**, uno de los bloques fundamentales de las redes neuronales.

El objetivo principal no es solo entrenar el modelo, sino realizar una **búsqueda de hiperparámetros** (Hyperparameter Tuning) para encontrar la configuración óptima que maximice la precisión de nuestro sistema de reconocimiento.

### Parámetros a optimizar:
1.  **Tasa de aprendizaje ($\eta$):** Controla cuánto se ajustan los pesos en cada actualización.
2.  **Épocas (*max_iter*):** Número de veces que el algoritmo revisa el dataset completo.
3.  **Regularización:** Evaluación del comportamiento con datos no vistos (Test set).

Ejecutar los siguientes comandos para importar las librerías necesarias y configurar el entorno de trabajo.

In [15]:
!pip install seaborn scikit-learn pandas pillow gradio matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\hecto\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


<p style="page-break-after:always;"></p>

# Perceptrón

In [16]:
#import warnings; warnings.filterwarnings("ignore")
import numpy as np

**PerceptronClassification:** Clasificación de muestras con una matriz de pesos. Todas las muestras deben empezar con un 1.

$$c(\boldsymbol{x}) = \operatorname*{argmax}_{c} g_{c}(\boldsymbol{x})\text{, con }g_{c}(\boldsymbol{x})=\boldsymbol{w}_{c}^{t}\boldsymbol{x}\text{ para todo }c$$

donde $\boldsymbol{x} = (1, x_1, ..., x_D)^t$, $\mathbf{W} = (\boldsymbol{w}_1, \boldsymbol{w}_2, ...,  \boldsymbol{w}_C)$ y $\boldsymbol{w}_c = (w_{c0}, w_{c1}, ..., w_{cD})^t$

In [17]:
def PerceptronClassification(X, W):
  Xh = np.hstack([np.ones((len(X), 1)), X])
  return np.argmax(Xh @ W, axis=1).reshape(-1, 1)

<p style="page-break-after:always;"></p>

**PerceptronTraining:** $\;$ Aprendizaje de una matriz de pesos $\mathbf{W}^*$ que minimize el número de errores de entrenamiento (con margen $b$)

$$\mathbf{W}^*=\operatorname*{argmin}_{\mathbf{W}=(\boldsymbol{w}_1,\dotsc,\boldsymbol{w}_C)}\sum_n\;\mathbb{ I}\biggl(\max_{c\neq y_n}\;\boldsymbol{w}_c^t\boldsymbol{x}_n+b \;>\; \boldsymbol{w}_{y_n}^t\boldsymbol{ x}_n\biggr)$$

Devuelve pesos en notación homogénea, $\mathbf{W}\in\mathbb{R}^{(1+D)\times C};\;$ junto con el número de errores y el número de iteraciones ejecutadas.

> **Entrada:** $\;$ datos $\;\mathcal{D}=\{(\boldsymbol{x}_n,y_n)\}\quad$ pesos $\;\mathbf{W}=\{\boldsymbol{w}_c\}\quad$ factor de aprendizaje $\;\alpha\in\mathbb{R}^{>0}\quad$ margen $\;b\in\mathbb{R}^{\geq 0}$ <br>
> **Salida:** $\;$ pesos optimizados $\;\mathbf{W}^*=\{\boldsymbol{w}_c\}^*$ <br>
> `repetir` <br>
>> `para cada` $\;$ muestra de entrenamiento $\,\boldsymbol{x}_n$ <br>
>>> *err* = `False` <br>
>>> `para cada` $\;$ clase $\,c\neq y_n$ <br>
>>>> `si` $\;\boldsymbol{w}_c^t\boldsymbol{x}_n+b>\boldsymbol{w}_{y_n}^t\boldsymbol{x}_n:\quad\boldsymbol{w}_c=\boldsymbol{w}_c-\alpha\boldsymbol{x}_n;\quad$ *err* = `True` <br>
>>>
>>> `si` $\;$ *err*: $\quad \boldsymbol{w}_{y_n}=\boldsymbol{w}_{y_n}+\alpha\boldsymbol{x}_n$
>
> `hasta que` $\;$ no haya errores de clasificación de entrenamiento

In [18]:
def PerceptronTraining(X, y, b=0.1, a=1.0, K=200):
    N, D = X.shape; Y = np.unique(y); C = Y.size; W = np.zeros((1+D, C))
    for k in range(1, K+1):  # para K iteraciones
        E = 0
        for n in range(N):  # para cada muestra de entrenamiento
            xn = np.array([1, *X[n, :]])
            cn = np.squeeze(np.where(Y==y[n]))  # Mapeamos a etiquetas de clase de 0 a C-1 (para simplificar)
            gn = W[:,cn].T @ xn; err = False
            for c in np.arange(C):  # para cada clase
                if c != cn and W[:,c].T @ xn + b >= gn:
                    W[:, c] = W[:, c] - a*xn; err = True
            if err:
                W[:, cn] = W[:, cn] + a*xn; E = E + 1
        if E == 0:
            break
    return W, E, k

In [19]:
import numpy as np

# ---------------------------------------------------------
# 1. Función de Clasificación (La fórmula matemática)
# ---------------------------------------------------------
def PerceptronClassification(X, W):
    # Añadimos la columna de 1s para el sesgo (bias)
    Xh = np.hstack([np.ones((len(X), 1)), X])
    # Producto matricial y argmax para elegir la clase ganadora
    return np.argmax(Xh @ W, axis=1).reshape(-1, 1)

# ---------------------------------------------------------
# 2. Función de Entrenamiento (El algoritmo)
# ---------------------------------------------------------
def PerceptronTraining(X, y, b=0.1, a=1.0, K=200):
    N, D = X.shape
    Y = np.unique(y)
    C = Y.size
    W = np.zeros((1+D, C)) # Inicializamos pesos a 0

    # Bucle principal de iteraciones (Épocas)
    for k in range(1, K+1):
        E = 0
        for n in range(N):
            # Vector de características con el 1 inicial
            xn = np.array([1, *X[n, :]])

            # Buscamos el índice de la clase correcta
            cn = np.squeeze(np.where(Y==y[n]))

            # Predicción actual
            gn = W[:, cn].T @ xn
            err = False

            # Comprobamos contra todas las otras clases
            for c in np.arange(C):
                if c != cn and W[:, c].T @ xn + b >= gn:
                    W[:, c] = W[:, c] - a*xn # Castigo a la clase incorrecta
                    err = True

            if err:
                W[:, cn] = W[:, cn] + a*xn   # Premio a la clase correcta
                E = E + 1

        if E == 0: # Si no hay errores, terminamos antes
            break

    return W, E, k

# Perceptrón aplicado al dataset MyDoodles 

**Lectura del dataset:** comprobamos también que matriz de datos y etiquetas tengan el número correcto de filas y columnas

In [20]:
import numpy as np

# 1. Cargar TU dataset (MyDoodles)
print("📂 Cargando dataset MyDoodles (16x16)...")
X = np.load("../data/doodles_X.npy")
y = np.load("../data/doodles_y.npy")

# 2. Verificaciones de seguridad (Estilo Profesional)
# Aseguramos que y sean enteros (0, 1, 2...) para que funcionen los índices de la fórmula
y = y.astype(int)

# 3. Normalización (Opcional pero recomendada para el Perceptrón)
# Como tus datos ya son 0s y 1s, no hace falta dividir por max,
# pero si tuvieras grises, haríamos: X = X / np.max(X)

print(f"✅ Datos listos: {X.shape[0]} muestras de {X.shape[1]} dimensiones (píxeles).")
print(f"   Clases detectadas: {np.unique(y)}")

📂 Cargando dataset MyDoodles (16x16)...
✅ Datos listos: 100 muestras de 256 dimensiones (píxeles).
   Clases detectadas: [0 1 2 3 4 5 6 7 8 9]


**Partición del dataset:** Creamos una partición del conjunto de datos Digits con un $20\%$ de los datos para test y el resto para entrenamiento, barajando previamente los datos según una semilla proporcionada por un generador de números aleatorios. Aquí, como en todo código que incluye aleatoriedad (que requiere generar números aleatorios), es conveniente fijar dicha semilla para poder reproducir los experimentos con precisión.

In [21]:
from sklearn.model_selection import train_test_split

# Configuración del experimento
TEST_SIZE = 0.2   # 20% para test
SEED = 23         # Semilla para que el azar sea siempre el mismo (reproducibilidad)

# División del dataset
# Añadimos 'stratify=y' para asegurar que haya la misma proporción de clases en ambos grupos
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    shuffle=True,
    random_state=SEED,
    stratify=y
)

print("✂️ Partición realizada (Train/Test Split):")
print(f"   • Entrenamiento (X_train): {X_train.shape}  -> Se usarán para calcular W")
print(f"   • Test (X_test):           {X_test.shape}  -> Se usarán para validar la precisión")

✂️ Partición realizada (Train/Test Split):
   • Entrenamiento (X_train): (80, 256)  -> Se usarán para calcular W
   • Test (X_test):           (20, 256)  -> Se usarán para validar la precisión


**Ajustamos el máximo número de iteraciones:**

In [22]:
import numpy as np

print("Ajustamos el número máximo de iteraciones:")
print("-" * 60)
print(f"{'alpha':^10} {'b':^10} {'K':^5} {'TrErr':^15} {'TeErr':^15}")
print("-" * 60)

b = 0.1
a = 1.0

# Probamos distintas iteraciones
for K in [1, 2, 5, 10, 20, 50, 100, 200]:
    # 1. Entrenar
    W, E, k = PerceptronTraining(X_train, y_train, b=b, a=a, K=K)

    # 2. Predecir
    y_test_pred = PerceptronClassification(X_test, W)

    # 3. Calcular Errores (¡CON EL FIX DE DIMENSIONES!)
    # .reshape(-1) asegura que ambos sean vectores planos (ej: de 20 elementos)
    mismatches = np.count_nonzero(y_test_pred.reshape(-1) != y_test.reshape(-1))
    err_test = mismatches / len(X_test)

    err_train = E / len(X_train)

    print(f"{a:^10.1e} {b:^10.1e} {K:^5d} {err_train:^15.1%} {err_test:^15.1%}")

print("-" * 60)

Ajustamos el número máximo de iteraciones:
------------------------------------------------------------
  alpha        b        K        TrErr           TeErr     
------------------------------------------------------------
 1.0e+00    1.0e-01     1        48.8%           35.0%     
 1.0e+00    1.0e-01     2        25.0%           20.0%     
 1.0e+00    1.0e-01     5        6.2%            5.0%      
 1.0e+00    1.0e-01    10        0.0%            20.0%     
 1.0e+00    1.0e-01    20        0.0%            20.0%     
 1.0e+00    1.0e-01    50        0.0%            20.0%     
 1.0e+00    1.0e-01    100       0.0%            20.0%     
 1.0e+00    1.0e-01    200       0.0%            20.0%     
------------------------------------------------------------


**Ajustamos el factor de aprendizaje (alpha):**

In [23]:
import numpy as np

print("Ajustamos el factor de aprendizaje (alpha):")
print("-" * 75)
print(f"{'alpha':^15} {'b':^10} {'K':^5} {'TrErr (Entreno)':^18} {'TeErr (Test)':^15}")
print("-" * 75)

# 1. Configuramos los fijos
# Usamos un K alto para dar tiempo a que los alphas pequeños aprendan
K_fijo = 200
b_fijo = 0.1   # Margen estándar (puedes probar con 100.0 si quieres ver la combinación)

# 2. Lista de Alphas a probar (Escala logarítmica)
lista_alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

for a_prueba in lista_alphas:

    # --- ENTRENAMIENTO ---
    W, E, k = PerceptronTraining(X_train, y_train, b=b_fijo, a=a_prueba, K=K_fijo)

    # --- PREDICCIÓN ---
    y_test_pred = PerceptronClassification(X_test, W)

    # --- CÁLCULO DE ERROR (Con el fix de reshape) ---
    mismatches = np.count_nonzero(y_test_pred.reshape(-1) != y_test.reshape(-1))

    err_test = mismatches / len(X_test)
    err_train = E / len(X_train)

    # --- IMPRIMIR FILA ---
    print(f"{a_prueba:^15.1e} {b_fijo:^10.1f} {K_fijo:^5d} {err_train:^18.1%} {err_test:^15.1%}")

print("-" * 75)
print("Interpretación: Elige el alpha con menor 'TeErr'.")
print("Nota: En el Perceptrón simple, a veces alpha solo escala los pesos")
print("y no cambia el resultado final tanto como el margen (b).")

Ajustamos el factor de aprendizaje (alpha):
---------------------------------------------------------------------------
     alpha          b        K    TrErr (Entreno)    TeErr (Test)  
---------------------------------------------------------------------------
    1.0e-03        0.1      200         0.0%             5.0%      
    1.0e-02        0.1      200         0.0%             10.0%     
    1.0e-01        0.1      200         0.0%             15.0%     
    1.0e+00        0.1      200         0.0%             20.0%     
    1.0e+01        0.1      200         0.0%             20.0%     
    1.0e+02        0.1      200         0.0%             20.0%     
    1.0e+03        0.1      200         0.0%             20.0%     
---------------------------------------------------------------------------
Interpretación: Elige el alpha con menor 'TeErr'.
Nota: En el Perceptrón simple, a veces alpha solo escala los pesos
y no cambia el resultado final tanto como el margen (b).


**Ajustamos el margen (b):**

In [24]:
print("Ajustamos el margen de separación (b):")
print("-" * 60)
print(f"{'alpha':^10} {'b':^10} {'K':^5} {'TrErr':^15} {'TeErr':^15}")
print("-" * 60)

# FIJAMOS alpha y K con valores que sabemos que funcionan
a = 1.0e-03
K = 50  # Ponemos suficientes iteraciones para dejarle aprender

# Probamos distintos márgenes (b)
# b=0.0 es el Perceptrón clásico sin margen
for b in [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]:

    # 1. Entrenar
    W, E, k = PerceptronTraining(X_train, y_train, b=b, a=a, K=K)

    # 2. Predecir
    y_test_pred = PerceptronClassification(X_test, W)

    # 3. Calcular Errores (CON EL FIX DE DIMENSIONES)
    mismatches = np.count_nonzero(y_test_pred.reshape(-1) != y_test.reshape(-1))

    err_test = mismatches / len(X_test)
    err_train = E / len(X_train)

    print(f"{a:^10.1e} {b:^10.1e} {K:^5d} {err_train:^15.1%} {err_test:^15.1%}")

print("-" * 60)
print("Interpretación: Un 'b' mayor suele generalizar mejor (menor TeErr).")

Ajustamos el margen de separación (b):
------------------------------------------------------------
  alpha        b        K        TrErr           TeErr     
------------------------------------------------------------
 1.0e-03    0.0e+00    50        0.0%            15.0%     
 1.0e-03    1.0e-02    50        0.0%            5.0%      
 1.0e-03    1.0e-01    50        0.0%            5.0%      
 1.0e-03    1.0e+00    50        31.2%           0.0%      
 1.0e-03    1.0e+01    50        98.8%           30.0%     
 1.0e-03    1.0e+02    50       100.0%           65.0%     
------------------------------------------------------------
Interpretación: Un 'b' mayor suele generalizar mejor (menor TeErr).


### Interpretación de Resultados

1.  **Separabilidad Lineal:** Observamos que el error de entrenamiento (`TrErr`) es **0.0%** en todos los casos. Esto confirma que nuestro dataset de *MyDoodles* es linealmente separable; el Perceptrón es capaz de aprenderse los dibujos de memoria perfectamente.

2.  **Impacto del Margen ($b$):**
    * Con márgenes pequeños ($b \le 0.1$), el modelo sufre de **sobreajuste (Overfitting)**: se aprende muy bien los datos de entreno, pero falla un **20%** en los datos nuevos (Test).
    * Al aumentar el margen ($b=10$, $b=100$), forzamos al modelo a dejar más espacio entre la frontera de decisión y los puntos. Esto mejora drásticamente la **generalización**.

3.  **Conclusión:** La mejor configuración para este dataset ha resultado ser un margen alto (**$b=100.0$**), logrando reducir el error en test hasta un **5.0%**. Esto indica que buscar una separación "ancha" ayuda a reconocer mejor los dibujos nuevos que no son exactamente iguales a los originales.

Guardar el modelo.

In [25]:
import os

print("🚀 Generando el modelo definitivo para la App...")

# 1. Configuración GANADORA (La que descubriste en la tabla)
b_final = 1.0e+00  # El margen que mejor funcionó
a_final = 1.0e-03    # Alpha estándar
K_final = 50     # Iteraciones suficientes

# 2. Entrenamos con TODO el dataset (X entero, no solo X_train)
#    Así el modelo tiene el máximo conocimiento posible.
W_definitivo, _, _ = PerceptronTraining(X, y, b=b_final, a=a_final, K=K_final)

# 3. Guardar el archivo de pesos
carpeta_modelos = "../models"
if not os.path.exists(carpeta_modelos):
    os.makedirs(carpeta_modelos)

ruta_modelo = os.path.join(carpeta_modelos, "perceptron_my_doodles.npy")
np.save(ruta_modelo, W_definitivo)

print(f"✅ ¡Éxito! Cerebro guardado en: {ruta_modelo}")
print("   Ahora tu App podrá cargar este archivo y adivinar tus dibujos.")

🚀 Generando el modelo definitivo para la App...
✅ ¡Éxito! Cerebro guardado en: ../models\perceptron_my_doodles.npy
   Ahora tu App podrá cargar este archivo y adivinar tus dibujos.


<p style="page-break-after:always;"></p>

<p style="page-break-after:always;"></p>

# 🔮 Prueba de Concepto: Clasificador en Tiempo Real

En esta sección final, pondremos a prueba nuestro modelo de **Perceptrón** en un entorno de producción simulado.

Cargaremos la matriz de pesos óptima ($W^*$) que hemos entrenado y guardado anteriormente. Utilizaremos una interfaz interactiva basada en **Gradio** para dibujar nuevos garabatos en tiempo real y verificar si el algoritmo es capaz de generalizar y reconocer nuestras intenciones artísticas.

**Instrucciones:**
1.  Ejecuta la celda de código.
2.  Dibuja una de las 10 formas en el panel izquierdo.
3.  Observa la predicción del modelo en el panel derecho.

**Recomendacion:** Realiza los dibujos grandes y claros para maximizar las posibilidades de reconocimiento correcto.

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image, ImageOps, ImageFilter # <--- OJO: Añadimos ImageFilter
import os

# 1. CARGA DEL MODELO
MODEL_PATH = "../models/perceptron_my_doodles.npy"
DOODLE_MAP = {
    0: "🙂 Carita Feliz", 1: "❤️ Corazón", 2: "⭐ Estrella", 3: "⚡ Rayo",
    4: "🌙 Luna",        5: "☁️ Nube",    6: "🌲 Árbol",   7: "🏠 Casa",
    8: "☹️ Carita Triste", 9: "☀️ Sol"
}

if os.path.exists(MODEL_PATH):
    W_loaded = np.load(MODEL_PATH)
    print("✅ Modelo cargado.")
else:
    raise FileNotFoundError("❌ No se encuentra el modelo.")

# 2. LÓGICA MEJORADA (Ahora con "Engrosamiento")
def predict_robust(image_input):
    if image_input is None: return "⚠️ Dibuja algo", None, "0 px"

    # --- A. Gestión de capas de Gradio ---
    if isinstance(image_input, dict):
        image = image_input.get("composite")
    else:
        image = image_input

    # --- B. Preprocesamiento INTELIGENTE ---
    # 1. Convertir a Gris e Invertir (Fondo negro, línea blanca)
    img = image.convert("L")
    img = ImageOps.invert(img)

    # 2. TRUCO DE MAGIA: ENGORDAR EL TRAZO (Dilatación) 🪄
    # Esto asegura que la línea no desaparezca al reducirla
    # Aplicamos un filtro que busca píxeles vecinos y los ilumina (MaxFilter)
    img = img.filter(ImageFilter.MaxFilter(9)) # Valor alto = Trazo muy gordo

    # 3. Redimensionar a 16x16
    img_tiny = img.resize((16, 16), resample=Image.BILINEAR)

    # 4. Binarizar (Limpieza final)
    x_array = np.array(img_tiny)
    x_bin = np.where(x_array > 50, 1, 0) # Umbral bajo para captar todo

    # --- C. Diagnóstico Visual ---
    pixels_activos = np.sum(x_bin)

    # Reconstruir imagen para que tú la veas
    debug_image = Image.fromarray((x_bin * 255).astype(np.uint8), mode="L")
    debug_image_big = debug_image.resize((200, 200), resample=Image.NEAREST)

    if pixels_activos < 5:
        return "⚠️ Trazo demasiado fino o vacío", debug_image_big, f"{pixels_activos} px"

    # --- D. Predicción Matemática ---
    x_flat = x_bin.flatten()
    x_aug = np.insert(x_flat, 0, 1) # Sesgo
    scores = x_aug @ W_loaded
    pred_idx = np.argmax(scores)

    # Confianza relativa (diferencia entre el mejor y el segundo mejor)
    sorted_scores = np.sort(scores)
    confidence = sorted_scores[-1] - sorted_scores[-2]

    prediction_text = f"🤖 PREDICCIÓN: {DOODLE_MAP.get(pred_idx, 'Desconocido')}\n(Confianza: {confidence:.2f})"

    return prediction_text, debug_image_big, f"Píxeles detectados: {pixels_activos}"

# 3. INTERFAZ
with gr.Blocks() as demo:
    gr.Markdown("## 🕵️‍♂️ Inspector de Perceptrón (Modo Robusto)")
    gr.Markdown("El sistema ahora **engorda** tu dibujo automáticamente antes de leerlo.")
    with gr.Row():
        with gr.Column():
            # Pincel por defecto muy grande
            inp = gr.Sketchpad(label="Dibuja AQUÍ", type="pil", brush=gr.Brush(default_size=20))
            btn = gr.Button("Adivinar", variant="primary")
        with gr.Column():
            out_txt = gr.Textbox(label="Resultado")
            out_img = gr.Image(label="Lo que ve la IA (Procesado)", type="pil")
            out_debug = gr.Label(label="Estadísticas")

    btn.click(fn=predict_robust, inputs=inp, outputs=[out_txt, out_img, out_debug])

demo.launch(debug=True)

✅ Modelo cargado.
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
